# Investing.com Data Preprocessing

**Time Index**

* Sort the index in ascending order before any fill operation.
* Reindex all datasets to a complete daily UTC calendar (2022-12-01 – 2026-09-16), so that they can be aligned with the other sources.

**Missing timestamps**

* Fill all missing days (weekends, holidays and irregular weekday gaps) with forward fill (FFILL) of `Price`. This carries over the last available market price and does not introduce artificial price movements, unlike interpolation. All gaps are short (up to 6 days) and no dataset has missing values at the start of the sample.

In [ ]:
import pandas as pd
import logging
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))
from src.config import DATA_DIR, DOWNLOAD_START_DATE, DOWNLOAD_END_DATE
from src.utils.data_quality import print_missing_values, print_index_summary

from src.utils.preprocessing import preprocess_investing_com
from src.utils.file_io import inspect_investing_files, load_parquet_data, save_data

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_inv = load_parquet_data(DATA_DIR / "clean" / "investing_com" / date_range)

In [4]:
data_inv = preprocess_investing_com(
    data_inv, end=pd.Timestamp(DOWNLOAD_END_DATE, tz="UTC")
)

## 1. Index Integrity – Confirmation

All five datasets now have a complete daily UTC index (2022-12-01 – 2026-09-16) in ascending order, with a single 1-day step and without duplicated timestamps.

In [5]:
for name, data in data_inv.items():
    print_index_summary(data=data, name=name)


--- [dutch_ttf_natural_gas_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  1 days 00:00:00: 1385

--- [eu_carbon_emissions_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  1 days 00:00:00: 1385

--- [eur_pln] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  1 days 00:00:00: 1385

--- [rotterdam_coal_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  1 days 00:00:00: 1385

--- [german_power_baseload_futures] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 0
Timezone:   UTC

Time steps:
  1 days 00:00:00: 1385


## 2. Missing Values – Confirmation

No missing values remain: the weekend, holiday and other missing days were filled with the last available `Price`.

In [6]:
for name, data in data_inv.items():
    print_missing_values(data=data, name=name)


--- [dutch_ttf_natural_gas_futures] Missing values ---
No missing values.

--- [eu_carbon_emissions_futures] Missing values ---
No missing values.

--- [eur_pln] Missing values ---
No missing values.

--- [rotterdam_coal_futures] Missing values ---
No missing values.

--- [german_power_baseload_futures] Missing values ---
No missing values.


## 3. Conclusions

The preprocessing decisions from the data quality audit have been applied to all five datasets, and the checks above confirm the result. The preprocessed datasets are saved to `data/preprocessed/investing_com/` and verified by reading the files back.

## 4. Saving Preprocessed Data

In [7]:
save_data(
    data_inv,
    DATA_DIR / "preprocessed" / "investing_com" / date_range,
)

INFO:src.utils.file_io:Saved preprocessed/investing_com/2022-12-01_2026-09-16/dutch_ttf_natural_gas_futures.parquet
INFO:src.utils.file_io:Saved preprocessed/investing_com/2022-12-01_2026-09-16/eu_carbon_emissions_futures.parquet
INFO:src.utils.file_io:Saved preprocessed/investing_com/2022-12-01_2026-09-16/eur_pln.parquet
INFO:src.utils.file_io:Saved preprocessed/investing_com/2022-12-01_2026-09-16/rotterdam_coal_futures.parquet
INFO:src.utils.file_io:Saved preprocessed/investing_com/2022-12-01_2026-09-16/german_power_baseload_futures.parquet


In [8]:
inspect_investing_files(
    DATA_DIR / "preprocessed" / "investing_com" / date_range,
)


--- Investing.com (5 file/s) ------------------------------
  [FILE] data/preprocessed/investing_com/2022-12-01_2026-09-16/dutch_ttf_natural_gas_futures.parquet | shape: (1386, 1) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00
  [FILE] data/preprocessed/investing_com/2022-12-01_2026-09-16/eu_carbon_emissions_futures.parquet | shape: (1386, 1) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00
  [FILE] data/preprocessed/investing_com/2022-12-01_2026-09-16/eur_pln.parquet | shape: (1386, 1) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00
  [FILE] data/preprocessed/investing_com/2022-12-01_2026-09-16/rotterdam_coal_futures.parquet | shape: (1386, 1) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00
  [FILE] data/preprocessed/investing_com/2022-12-01_2026-09-16/german_power_baseload_futures.parquet | shape: (1386, 1) | idx: 2022-12-01 00:00:00+00:00-2026-09-16 00:00:00+00:00
